# Show influence of staining on contrast
The notebook started out as a copy from https://github.com/habi/FemalePelvicFloor/blob/main/Preview_Fetal_Scans.ipynb
We want to show the influence of (Lugol) staining on contrast in a sample.

In [ ]:
import platform
import os
import glob
import pandas
import dask
from dask.distributed import Client, LocalCluster
import dask_image.imread
import matplotlib.pyplot as plt
from matplotlib_scalebar.scalebar import ScaleBar
from matplotlib.gridspec import GridSpec
import seaborn
import numpy
from tqdm.auto import tqdm, trange
import imageio
import skimage
from scipy.signal import find_peaks

In [ ]:
# Import our own parsing functions which we've added as submodule
from BrukerSkyScanLogfileRuminator.parsing_functions import *

In [ ]:
# Set dask temporary folder
# Do this before creating a client: https://stackoverflow.com/a/62804525/323100
import tempfile
if 'Linux' in platform.system():
    tmp = os.path.join(os.sep, 'media', 'habi', 'Fast_SSD')
elif 'Darwin' in platform.system():
    tmp = tempfile.gettempdir()
else:
    if 'anaklin' in platform.node():
        tmp = os.path.join('F:\\')
    else:
        tmp = os.path.join('D:\\')
dask.config.set({'temporary_directory': os.path.join(tmp, 'tmp')})
print('Dask temporary files go to %s' % dask.config.get('temporary_directory'))

In [ ]:
# Start cluster and client now, after setting tempdir
cluster = LocalCluster()
client = Client(cluster)

In [ ]:
print('You can seee what DASK is doing at "http://localhost:%s/status"' % client.scheduler_info()['services']['dashboard'])

In [ ]:
seaborn.set_context("talk")

In [ ]:
# Set up figure defaults
plt.rc('image', cmap='gray', interpolation='nearest')  # Display all images in b&w and with 'nearest' interpolation
plt.rcParams['figure.dpi'] = 300

In [ ]:
# Setup scale bar defaults
plt.rcParams['scalebar.location'] = 'lower right'
plt.rcParams['scalebar.frameon'] = False
plt.rcParams['scalebar.color'] = 'white'

In [ ]:
# Display all plots identically
lines = 3
# And then do something like
# plt.subplot(lines, int(numpy.ceil(len(Data) / float(lines))), c + 1)

In [ ]:
# Different locations if running either on Linux or Windows
FastSSD = False
# to speed things up significantly
if 'Linux' in platform.system():
    if FastSSD:
        BasePath = os.path.join(os.sep, 'media', 'habi', 'Fast_SSD')
    else:
        BasePath = os.path.join(os.sep, 'home', 'habi', 'research_storage_djonov')
elif 'Darwin' in platform.system():
    # First mount smb://resstore.unibe.ch/ana_rs_djonov/data in the Finder
    FastSSD = False
    BasePath = os.path.join('/Volumes/data/')
elif 'Windows' in platform.system():
    if FastSSD:
        BasePath = os.path.join('F:\\')
    else:
        if 'anaklin' in platform.node():
            BasePath = os.path.join('V:\\')
        else:
            BasePath = os.path.join('V:\\')
Root = os.path.join(BasePath, 'Aaldijk', 'PelvicFloor')
print('We are loading all the data from %s' % Root)

In [ ]:
# Make us a dataframe for saving all that we need
Data = pandas.DataFrame()

In [ ]:
# Get *all* log files
Data['LogFile'] = [f for f in sorted(glob.glob(os.path.join(Root, '**', '*.log'), recursive=True))]

In [ ]:
# Use only the mouse scans for the lecture
for c, row in Data.iterrows():
    if 'Mouse01' not in row.LogFile:
        Data.drop([c], inplace=True)
Data.reset_index(inplace=True)

In [ ]:
# Get all folders
Data['Folder'] = [os.path.dirname(f) for f in Data['LogFile']]

In [ ]:
# Get rid of all logfiles that we don't want
for c, row in Data.iterrows():
    if 'rec' not in row.Folder:  # drop all non-rec folders
        Data.drop([c], inplace=True)
    elif 'SubScan' in row.Folder:  # drop all partial reconstructions which might be there from synchronization
        Data.drop([c], inplace=True)
    elif 'rectmp.log' in row.LogFile:  # drop all temporary logfiles
        Data.drop([c], inplace=True)
# Reset dataframe to something that we would get if we only would have loaded the 'rec' files
Data = Data.reset_index(drop=True)

In [ ]:
# Generate us some meaningful colums
Data['Sample'] = [log[len(Root) + 1:].split(os.sep)[0] for log in Data['LogFile']]
Data['SampleName'] = [sn.split('_')[0] for sn in Data['Sample']]
Data['Scan'] = ['_'.join(log[len(Root) + 1:].split(os.sep)[1:-1]) for log in Data['LogFile']]

In [ ]:
# Get the file names of the reconstructions
Data['Reconstructions'] = [sorted(glob.glob(os.path.join(f, '*rec0*.png'))) for f in Data['Folder']]
Data['Number of reconstructions'] = [len(r) for r in Data.Reconstructions]

In [ ]:
# Get scanning parameters to doublecheck from logfiles
Data['Scanner'] = [scanner(log) for log in Data['LogFile']]
Data['Voltage'] = [voltage(log) for log in Data['LogFile']]
Data['Current'] = [current(log) for log in Data['LogFile']]
Data['Voxelsize'] = [pixelsize(log, rounded=True) for log in Data['LogFile']]
Data['CameraWindow'] = [projection_size(log) for log in Data['LogFile']]
Data['Exposuretime'] = [exposuretime(log) for log in Data['LogFile']]
Data['Averaging'] = [averaging(log) for log in Data['LogFile']]
Data['Stacks'] = [stacks(log) for log in Data['LogFile']]
Data['RotationStep'] = [rotationstep(log) for log in Data['LogFile']]
Data['Scan date'] = [scandate(log) for log in Data['LogFile']]
Data['Scan time'] = [duration(log) for log in Data['LogFile']]

In [ ]:
# Get reconstruction parameters to doublecheck from logfiles
Data['Grayvalue'] = [reconstruction_grayvalue(log) for log in Data['LogFile']]
Data['RingartefactCorrection'] = [ringremoval(log) for log in Data['LogFile']]
Data['BeamHardeningCorrection'] = [beamhardening(log) for log in Data['LogFile']]
Data['DefectPixelMasking'] = [defectpixelmasking(log) for log in Data['LogFile']]
Data['ROI'] = [region_of_interest(log) for log in Data['LogFile']]

In [ ]:
# Sort by scan date
Data.sort_values(by='Scan date', inplace=True, ignore_index=True)

In [ ]:
# Calculate time 'spent' since start
Data['Time passed'] = [sd - Data['Scan date'].min() for sd in Data['Scan date']]
# Also extract days, rounded
Data['Days passed'] = [t.round('d') for t in Data['Time passed']]

In [ ]:
# Show everything
Data[['Sample', 'Scan date', 'Voxelsize', 'Time passed', 'Days passed', 'Folder']]

In [ ]:
# Drop all mouse scans *not* scanned at 15 um
Data = Data[Data.Voxelsize == 15]
Data = Data.reset_index(drop=True)

In [ ]:
# Use only first, middle and last scan of the mouse series for the stuff below
Data = Data.iloc[[0, len(Data) // 2, -1]].reset_index(drop=True)

In [ ]:
Data

In [ ]:
Data['PreviewImagePath'] = [sorted(glob.glob(os.path.join(f, '*_spr.bmp')))[0] for f in Data['Folder']]
Data['PreviewImage'] = [dask_image.imread.imread(pip).squeeze()
                        if pip
                        else numpy.random.random((100, 100)) for pip in Data['PreviewImagePath']]

In [ ]:
# Make an approximately square overview image
lines = 1

In [ ]:
Data['Days passed'][1].days

In [ ]:
# for c, row in Data.iterrows():
#     plt.subplot(lines, int(numpy.ceil(len(Data) / float(lines))), c + 1)
#     plt.imshow(row.PreviewImage.squeeze())
#     plt.title('%s: Preview after %s days' % (row['SampleName'], row['Days passed'].days))
#     plt.gca().add_artist(ScaleBar(row['Voxelsize'],
#                                   'um',
#                                   color='black',
#                                   frameon=True))
#     plt.axis('off')
# plt.tight_layout()
# plt.savefig(os.path.join(Root, 'ScanOverviews.Foetus.png'),
#             bbox_inches='tight')
# plt.show()

In [ ]:
# Load all reconstructions into ephemereal DASK arrays
Reconstructions = [None] * len(Data)
for c, row in tqdm(Data.iterrows(),
                   desc='Load reconstructions',
                   total=len(Data)):
    Reconstructions[c] = dask_image.imread.imread(os.path.join(row['Folder'],
                                                               '*rec*.png'))

In [ ]:
# How big are the datasets?
Data['Size'] = [rec.shape for rec in Reconstructions]

In [ ]:
# The three cardinal directions
directions = ['Axial',
              'Coronal',
              'Sagittal']

In [ ]:
# Read or calculate the middle slices, put them into the dataframe and save them to disk
for d, direction in enumerate(directions):
    Data['Mid_' + direction] = [None] * len(Reconstructions)
for c, row in tqdm(Data.iterrows(), desc='Middle images', total=len(Data), leave=False):
    for d, direction in tqdm(enumerate(directions),
                             desc='%s/%s' % (row['Sample'], row['Scan']),
                             leave=False,
                             total=len(directions)):
        outfilepath = os.path.join(os.path.dirname(row['Folder']),
                                   '%s.%s.Middle.%s.png' % (row['Sample'],
                                                            row['Scan'],
                                                            direction))
        if not os.path.exists(outfilepath):
            # Generate requested axial view
            if 'Axial' in direction:
                Data.at[c, 'Mid_' + direction] = Reconstructions[c][Data['Size'][c][0] // 2].compute().squeeze()
            if 'Coronal' in direction:
                Data.at[c, 'Mid_' + direction] = Reconstructions[c][:, Data['Size'][c][1] // 2, :].compute().squeeze()
            if 'Sagittal' in direction:
                Data.at[c, 'Mid_' + direction] = Reconstructions[c][:, :, Data['Size'][c][2] // 2].compute().squeeze()
            # Save the calculated 'direction' view to disk
            imageio.imwrite(outfilepath, (Data.at[c, 'Mid_' + direction]))
        Data.at[c, 'Mid_' + direction] = dask_image.imread.imread(outfilepath).squeeze()

In [ ]:
# Make output directory for lecture images
outputdir = 'ContrastAndStaining'
os.makedirs(outputdir, exist_ok=True)

In [ ]:
for c, row in tqdm(Data.iterrows(),
                   desc='Saving middle images overview',
                   total=len(Data),
                   leave=False):
    outfilepath = os.path.join(outputdir,
                               '%s.%03d.MiddleSlices.png' % (row['SampleName'], row['Days passed'].days))
    fig = plt.figure(figsize=(16, 9 / 2))
    gs = GridSpec(1, len(directions), figure=fig)
    for d, direction in enumerate(directions):
        ax = fig.add_subplot(gs[0, d])
        img = skimage.exposure.equalize_adapthist(row['Mid_' + direction].compute())
        ax.imshow(img, cmap='gray', aspect='equal')
        ax.add_artist(ScaleBar(row['Voxelsize'], 'um'))
        ax.set_xticks([])
        ax.set_yticks([])
        if d == 0:
            ax.set_ylabel(f"{row.SampleName}\nStained for {row['Days passed'].days} d")
    plt.tight_layout()
    plt.savefig(outfilepath, transparent=True)
    plt.show()

In [ ]:
# Read or calculate the directional MIPs, put them into the dataframe and save them to disk
for d, direction in enumerate(directions):
    Data['MIP_' + direction] = [None] * len(Reconstructions)
for c, row in tqdm(Data.iterrows(), desc='MIPs', total=len(Data), leave=False):
    for d, direction in tqdm(enumerate(directions),
                             desc='%s/%s' % (row['Sample'], row['Scan']),
                             leave=False,
                             total=len(directions)):
        outfilepath = os.path.join(os.path.dirname(row['Folder']),
                                   '%s.%s.MIP.%s.png' % (row['Sample'],
                                                         row['Scan'],
                                                         direction))
        # Generate and write out MIP if it doesn't exist on disk
        if not os.path.exists(outfilepath):
            Data.at[c, 'MIP_' + direction] = Reconstructions[c].max(axis=d).compute().squeeze()
            imageio.imwrite(outfilepath, Data.at[c, 'MIP_' + direction].astype('uint8'))
        Data.at[c, 'MIP_' + direction] = dask_image.imread.imread(outfilepath).squeeze()

In [ ]:
# Show middle slices
for c, row in tqdm(Data.iterrows(),
                   desc='Saving MIP images overview',
                   total=len(Data),
                   leave=False):
    outfilepath = os.path.join(outputdir,
                               '%s.%03d.MIPs.png' % (row['SampleName'], row['Days passed'].days))
    fig = plt.figure(figsize=(16, 9 / 2))
    gs = GridSpec(1, 3)    
    for d, direction in enumerate(directions):
        ax = fig.add_subplot(gs[0, d])
        img = skimage.exposure.equalize_adapthist(row['MIP_' + direction].compute())
        ax.imshow(img, cmap='gray', aspect='equal')
        ax.add_artist(ScaleBar(row['Voxelsize'], 'um'))
        ax.set_xticks([])
        ax.set_yticks([])
        if d == 0:
            ax.set_ylabel(f"{row.SampleName}\nStained for {row['Days passed'].days} d")
    plt.tight_layout()
    plt.savefig(outfilepath, transparent=True)
    plt.show()

In [ ]:
# Extract line profile of images to show later
# Define two points (row, col)
start = (600, 400)
stop = (1200, 1700)
width=10
# Extract intensity profile
Data['Lineprofile'] = [skimage.measure.profile_line(img, start, stop, linewidth=width) for img in Data['Mid_Axial']]

In [ ]:
for c, row in tqdm(Data.iterrows(),
                   desc='Saving middle images overview',
                   total=len(Data),
                   leave=False):
    outfilepath = os.path.join(outputdir,
                               '%s.%03d.MiddleSlices.Lineprofile.png' % (row['SampleName'], row['Days passed'].days))
    fig = plt.figure(figsize=(16, 9 / 2))
    ax_img = fig.add_subplot(gs[0, 0])
    img = skimage.exposure.equalize_adapthist(row['Mid_Axial'].compute())
    ax_img.imshow(img, cmap='gray', aspect='equal')
    ax_img.plot([start[1], stop[1]], [start[0], stop[0]], '--', c='white')
    ax_img.add_artist(ScaleBar(row['Voxelsize'], 'um'))
    ax_img.set_ylabel(f"{row.SampleName}\nStained for {row['Days passed'].days} d")
    ax_img.set_xticks([])
    ax_img.set_yticks([])    
    ax_profile = fig.add_subplot(gs[0, 1:3])
    ax_profile.plot(row['Lineprofile'], c='#e4003c')
    ax_profile.set_ylim([0, 150])
    ax_profile.set_xticks([])
    ax_profile.set_yticks([])
    seaborn.despine(ax=ax_profile)
    plt.tight_layout()
    plt.savefig(outfilepath, transparent=True)
    plt.show()